In [1]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.neural_network import MLPRegressor
from sklearn.base import BaseEstimator
from typing import Callable
from sklearn.metrics import mean_absolute_error, accuracy_score, mean_absolute_percentage_error, r2_score, root_mean_squared_error
import pandas as pd
import bambi as bmb
import arviz as az
from scipy.stats import norm
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# custom 
from utils import ts_parents, all_formulas_from_structures, train_regime_models_bayesian, classify_regime_bayesian, sliding_window_regime_prediction

In [ ]:
def forecasting_metrics(y_test, y_pred):
    return {
        "R-squared (R2)": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
    }


def prepare_train_sets(data, causal_structures, train_n=500):
    train = data[:train_n]
    train_by_regime = {
        regime: train[train["regime"] == regime].drop(columns=["regime"])
        for regime in causal_structures
    }
    return {
        regime: ts_parents(train_by_regime[regime], causal_structures[regime])
        for regime in causal_structures
    }


def prepare_classification_sets(data, causal_structures, train_n=500, w=5):
    test = data[train_n - w - 1:]
    true_regime = test["regime"].iloc[w:].to_numpy()
    test_data = test.drop(columns=["regime"])

    X_test = {
        regime: ts_parents(test_data, causal_structures[regime])
        for regime in causal_structures
    }
    return X_test, true_regime


def fit_bayesian_regime_pipeline(data, causal_structures, train_n=500, w=5):
    X_train = prepare_train_sets(data, causal_structures, train_n=train_n)
    all_formulas = all_formulas_from_structures(causal_structures)

    bayesian_models = train_regime_models_bayesian(
        X=X_train,
        all_formulas=all_formulas
    )

    X_test, true_regime = prepare_classification_sets(
        data,
        causal_structures,
        train_n=train_n,
        w=w
    )

    regime_classification, errors = classify_regime_bayesian(
        X_test=X_test,
        all_formulas=all_formulas,
        bayesian_models=bayesian_models,
        causal_structures=causal_structures
    )

    regime_acc_w1 = accuracy_score(regime_classification[w - 1:], true_regime)
    window_preds = sliding_window_regime_prediction(errors, window_size=w)
    regime_acc_w5 = accuracy_score(window_preds, true_regime)

    return bayesian_models, window_preds, regime_acc_w1, regime_acc_w5


def forecast_bayesian_regime_x2(data, causal_structures, bayesian_models, window_preds, train_n=500):
    test_data = data[train_n - 1:].copy()
    test_data.loc[:, "regime"] = window_preds

    test = ts_parents(test_data, causal_structures[0])

    test_features = {
        regime: test[test["regime"] == regime].drop(columns=["regime"])
        for regime in causal_structures
    }

    preds_by_regime = {}
    for regime in causal_structures:
        model = bayesian_models[regime]["X2"]["model"]
        idata = bayesian_models[regime]["X2"]["idata"]

        preds = model.predict(
            idata=idata,
            data=test_features[regime],
            kind="response",
            inplace=False
        ).posterior_predictive["X2"]

        preds_by_regime[regime] = preds.mean(dim=("chain", "draw")).values

    # need to be in correct order 
    y_pred = np.concatenate([preds_by_regime[1], preds_by_regime[0]])
    y_test = test["X2"].values

    return y_test, y_pred


def forecast_markov_switching_x2(data, train_n=500):
    df = data.copy()
    n_vars = len(df.columns)

    for j in range(n_vars):
        df[f"X{j}_lag1"] = df[f"X{j}"].shift(1)

    df = df.dropna().reset_index(drop=True)

    train = df.iloc[:train_n - 1]
    test = df.iloc[train_n - 1:700]

    y_train = train["X2"]
    exog_cols = [f"X{j}_lag1" for j in range(n_vars)]
    exog_train = train[exog_cols]

    model = MarkovRegression(
        endog=y_train,
        k_regimes=2,
        exog=exog_train,
        trend="c",
        switching_trend=True,
        switching_exog=True,
        switching_variance=True
    )
    res = model.fit(em_iter=10, search_reps=5, disp=False)

    p00, p10 = res.params["p[0->0]"], res.params["p[1->0]"]
    P = np.array([[p00, 1 - p00],
                  [p10, 1 - p10]])

    intercepts = np.array([res.params["const[0]"], res.params["const[1]"]])

    betas = np.vstack([
        [res.params[f"x{k+1}[0]"], res.params[f"x{k+1}[1]"]]
        for k in range(n_vars)
    ]).T

    sigmas = np.sqrt([res.params["sigma2[0]"], res.params["sigma2[1]"]])

    pi = res.filtered_marginal_probabilities.iloc[-1].values
    y_pred = []

    for _, row in test.iterrows():
        pi_pred = pi.dot(P)
        lag_vals = np.array([row[c] for c in exog_cols])
        mus = intercepts + (betas * lag_vals).sum(axis=1)
        y_hat = pi_pred.dot(mus)
        y_pred.append(y_hat)

        likelihoods = norm.pdf(row["X2"], loc=mus, scale=sigmas)
        pi = pi_pred * likelihoods
        pi /= pi.sum()

    y_test = test["X2"].values
    return y_test, np.array(y_pred)


def forecast_var_x2(data, train_n=500):
    df = data.copy()

    train = df.iloc[:train_n]
    test = df.iloc[train_n:700]

    model = VAR(train)
    res = model.fit(maxlags=1)

    lag_order = res.k_ar
    history = train.values.copy()
    y_pred = []

    for obs in test.values:
        last_obs = history[-lag_order:]
        yhat = res.forecast(y=last_obs, steps=1)
        y_pred.append(yhat[0, 2])
        history = np.vstack([history, obs])

    y_test = test["X2"].values
    return y_test, np.array(y_pred)


def evaluate_simulation(simulation, train_n=500, w=5):
    data = simulation["df"].copy()
    causal_structures = simulation["causal_structure"]

    bayesian_models, window_preds, regime_acc_w1, regime_acc_w5 = fit_bayesian_regime_pipeline(
        data=data,
        causal_structures=causal_structures,
        train_n=train_n,
        w=w
    )

    y_test_bayes, y_pred_bayes = forecast_bayesian_regime_x2(
        data=data,
        causal_structures=causal_structures,
        bayesian_models=bayesian_models,
        window_preds=window_preds,
        train_n=train_n
    )

    y_test_msv, y_pred_msv = forecast_markov_switching_x2(
        data=data.drop(columns=["regime"]),
        train_n=train_n
    )

    y_test_var, y_pred_var = forecast_var_x2(
        data=data.drop(columns=["regime"]),
        train_n=train_n
    )

    # --- regime accuracy rows ---
    regime_rows = [
        {
            "seed": simulation.get("seed"),
            "model": "Bayesian Regime Model",
            "w=1": regime_acc_w1,
            "w=10": regime_acc_w5,
        }
    ]

    # --- forecasting rows ---
    forecast_rows = []

    row_bayes = {
        "seed": simulation.get("seed"),
        "model": "Bayesian Regime Model",
    }
    row_bayes.update(forecasting_metrics(y_test_bayes, y_pred_bayes))
    forecast_rows.append(row_bayes)

    row_msv = {
        "seed": simulation.get("seed"),
        "model": "Markov-Switching VAR",
    }
    row_msv.update(forecasting_metrics(y_test_msv, y_pred_msv))
    forecast_rows.append(row_msv)

    row_var = {
        "seed": simulation.get("seed"),
        "model": "VAR(1)",
    }
    row_var.update(forecasting_metrics(y_test_var, y_pred_var))
    forecast_rows.append(row_var)

    return regime_rows, forecast_rows

In [3]:
import pickle
with open("mc_system3.pkl", "rb") as f:
    causal_discovery_results = pickle.load(f)

len(causal_discovery_results)

5

In [4]:
regime_rows_all = []
forecast_rows_all = []
i = 1
for simulation in causal_discovery_results:
    print(f"forecasting monte carlo {i}...")
    regime_rows, forecast_rows = evaluate_simulation(simulation, train_n=500, w=10)
    regime_rows_all.extend(regime_rows)
    forecast_rows_all.extend(forecast_rows)
    i += 1

regime_df = pd.DataFrame(regime_rows_all)
forecast_df = pd.DataFrame(forecast_rows_all)

forecasting monte carlo 1...
forecasting monte carlo 2...
forecasting monte carlo 3...
forecasting monte carlo 4...
forecasting monte carlo 5...


## Forecasting Results

In [ ]:
# full results 
forecast_df

,seed,model,R-squared (R2),MAE,RMSE,MAPE
0,3,Bayesian Regime Model,0.569714,0.774411,0.971407,1.617079
1,3,Markov-Switching VAR,0.551203,0.784319,0.992082,1.575602
2,3,VAR(1),0.367299,0.965891,1.177937,2.630678
3,11,Bayesian Regime Model,0.520297,0.853405,1.047110,2.205186
4,11,Markov-Switching VAR,0.483526,0.883852,1.086501,2.317325
5,11,VAR(1),0.315637,0.994790,1.250689,1.757155
6,17,Bayesian Regime Model,0.470355,0.839391,1.038367,2.290674
7,17,Markov-Switching VAR,0.451424,0.847914,1.056761,2.215389
8,17,VAR(1),0.275443,0.982674,1.214492,2.763835
9,24,Bayesian Regime Model,0.413600,0.832492,1.039499,3.266087


In [ ]:
# test set regime classification
regime_df.rename(columns={"w=35":"w=10"}, inplace=True) # should be w=10
regime_df[["w=1","w=10"]].agg(["mean", "std"])

,w=1,w=10
mean,0.790050,0.977114
std,0.040935,0.014333


In [10]:
# summary with mean and standard deviation over monte carlo simulations
forecast_summary = forecast_df.drop(columns=["seed"]).groupby("model").agg(["mean", "std"])
forecast_summary

R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.491060  0.058234  0.823118  0.030392  1.027104   
Markov-Switching VAR        0.438047  0.107879  0.855658  0.052498  1.075800   
VAR(1)                      0.301239  0.042241  0.971157  0.027810  1.205697   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.031336  2.615339  0.846325  
Markov-Switching VAR   0.062936  2.626506  0.857135  
VAR(1)                 0.033275  2.730579  0.648241

In [11]:
forecast_summary.to_latex()

'\\begin{tabular}{lrrrrrrrr}\n\\toprule\n & \\multicolumn{2}{r}{R-squared (R2)} & \\multicolumn{2}{r}{MAE} & \\multicolumn{2}{r}{RMSE} & \\multicolumn{2}{r}{MAPE} \\\\\n & mean & std & mean & std & mean & std & mean & std \\\\\nmodel &  &  &  &  &  &  &  &  \\\\\n\\midrule\nBayesian Regime Model & 0.491060 & 0.058234 & 0.823118 & 0.030392 & 1.027104 & 0.031336 & 2.615339 & 0.846325 \\\\\nMarkov-Switching VAR & 0.438047 & 0.107879 & 0.855658 & 0.052498 & 1.075800 & 0.062936 & 2.626506 & 0.857135 \\\\\nVAR(1) & 0.301239 & 0.042241 & 0.971157 & 0.027810 & 1.205697 & 0.033275 & 2.730579 & 0.648241 \\\\\n\\bottomrule\n\\end{tabular}\n'